In [1]:
!python --version
!nvcc --version
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Python 3.12.13
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpms19mepx".


In [2]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv

name, compute_cap
Tesla T4, 7.5


## Rozwiązanie

In [3]:
!nvcc -o matrix_transpose matrix_transpose.cu
!./matrix_transpose

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Execution Times for matrix_transpose.cu (Avg 20 iterations)
Matrix Size: 1024 x 1024
 [Ex 1.1] Naive (Global Only)     : 0.194 ms
 [Ex 1.1] Shared (Bank Conflicts) : 0.137 ms
 [Ex 1.3] Direct Copy (Coalesced) : 0.051 ms
----------------------------------------------------
Matrix Size: 2048 x 2048
 [Ex 1.1] Naive (Global Only)     : 0.742 ms
 [Ex 1.1] Shared (Bank Conflicts) : 0.516 ms
 [Ex 1.3] Direct Copy (Coalesced) : 0.180 ms
----------------------------------------------------
Matrix Size: 4096 x 4096
 [Ex 1.1] Naive (Global Only)     : 2.927 ms
 [Ex 1.1] Shared (Bank Conflicts) : 2.045 ms
 [Ex 1.3] Direct Copy (Coalesced) : 0.700 ms
----------------------------------------------------


In [4]:
!ncu --set basic --section Occupancy ./matrix_transpose

Streaming output truncated to the last 5000 lines.
    Achieved Occupancy                        %        88.46
    Achieved Active Warps Per SM           warp        28.31
    ------------------------------- ----------- ------------

    OPT   Est. Local Speedup: 11.54%                                                                                    
          The difference between calculated theoretical (100.0%) and measured achieved occupancy (88.5%) can be the     
          result of warp scheduling overheads or workload imbalances during the kernel execution. Load imbalances can   
          occur between warps within a block as well as across blocks of the same kernel. See the CUDA Best Practices   
          Guide (https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/index.html#occupancy) for more details on     
          optimizing occupancy.                                                                                         

    Section: GPU and Memory Workload D

In [5]:
!nvcc -o conflict_solved conflict_solved.cu
!./conflict_solved

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Execution Times for conflict_solved.cu (Avg 20 Iterations):
Matrix Size: 1024 x 1024
 [Ex 1.2] Naive (Global Only)     : 0.194 ms
 [Ex 1.2] Shared (Conflict Solved): 0.059 ms
----------------------------------------------------
Matrix Size: 2048 x 2048
 [Ex 1.2] Naive (Global Only)     : 0.743 ms
 [Ex 1.2] Shared (Conflict Solved): 0.213 ms
----------------------------------------------------
Matrix Size: 4096 x 4096
 [Ex 1.2] Naive (Global Only)     : 2.926 ms
 [Ex 1.2] Shared (Conflict Solved): 0.830 ms
----------------------------------------------------


In [6]:
!ncu --set basic --section Occupancy ./conflict_solved

Streaming output truncated to the last 5000 lines.
    Metric Name                Metric Unit Metric Value
    -------------------------- ----------- ------------
    Average DRAM Active Cycles       cycle    1,405,084
    Total DRAM Elapsed Cycles        cycle   29,548,544
    Average L1 Active Cycles         cycle   413,020.15
    Total L1 Elapsed Cycles          cycle   17,321,536
    Average L2 Active Cycles         cycle   629,499.81
    Total L2 Elapsed Cycles          cycle   20,206,592
    Average SM Active Cycles         cycle   413,020.15
    Total SM Elapsed Cycles          cycle   17,321,536
    Average SMSP Active Cycles       cycle   388,487.52
    Total SMSP Elapsed Cycles        cycle   69,286,144
    -------------------------- ----------- ------------

  matrix_transpose_naive(int *, int *, int) (64, 64, 1)x(32, 32, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name  